In [5]:
import csv

# Step 1: Extract sequence IDs with index from .fna
def extract_indexed_sequence_ids(fna_file):
    indexed_ids = []
    with open(fna_file, 'r') as file:
        index = 1
        for line in file:
            if line.startswith('>'):
                seq_id = line.split()[0][1:]  # Remove '>'
                indexed_ids.append((index, seq_id))
                index += 1
    return indexed_ids

# Step 2: Map sequence ID -> virus tax id and pmid
def map_seq_to_taxid_pmid(tsv_file, sequence_ids_set):
    seq_to_info = {}
    with open(tsv_file, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter='\t')
        for row in reader:
            refseq_id = row['refseq id'].strip()
            if refseq_id in sequence_ids_set:
                virus_tax_id = row['virus tax id'].strip()
                pmid = row['pmid'].strip()
                seq_to_info[refseq_id] = {
                    'virus_tax_id': virus_tax_id,
                    'pmid': pmid
                }
    return seq_to_info

# Step 3: Map virus tax id -> ftp_path from assembly_summary_viral.txt
def map_taxid_to_ftp(assembly_file):
    taxid_to_ftp = {}
    with open(assembly_file, 'r') as file:
        for line in file:
            if line.startswith('#'):
                continue
            parts = line.strip().split('\t')
            if len(parts) > 19:
                taxid = parts[5].strip()
                ftp_path = parts[19].strip()
                taxid_to_ftp[taxid] = ftp_path
    return taxid_to_ftp

# Step 4: Generate summary
def generate_summary(indexed_ids, seq_to_info, taxid_to_ftp, output_file):
    with open(output_file, 'w', newline='') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerow(['Index', 'Sequence ID', 'Virus Tax ID', 'FTP Path'])
        for index, seq_id in indexed_ids:
            info = seq_to_info.get(seq_id)
            if info:
                tax_id = info['virus_tax_id']
                ftp_path = taxid_to_ftp.get(tax_id, 'N/A')
            else:
                tax_id = 'N/A'
                ftp_path = 'N/A'
            writer.writerow([index, seq_id, tax_id, ftp_path])

# Main Execution
fna_file = 'virushostdb.genomic.fna'
tsv_file = 'virushostdb.tsv'
assembly_file = 'assembly_summary_viral.txt'
output_file = 'summary_output.tsv'

indexed_ids = extract_indexed_sequence_ids(fna_file)
sequence_ids = {seq_id for _, seq_id in indexed_ids}
seq_to_info = map_seq_to_taxid_pmid(tsv_file, sequence_ids)
taxid_to_ftp = map_taxid_to_ftp(assembly_file)
generate_summary(indexed_ids, seq_to_info, taxid_to_ftp, output_file)

print(f"Summary file written to {output_file}")


Summary file written to summary_output.tsv


In [6]:
def analyze_summary(summary_file):
    total = 0
    with_ftp = 0

    with open(summary_file, 'r') as file:
        next(file)  # skip header
        for line in file:
            total += 1
            parts = line.strip().split('\t')
            ftp_path = parts[3]
            if ftp_path != 'N/A':
                with_ftp += 1

    print(f"Total entries: {total}")
    print(f"Entries with FTP link: {with_ftp}")
    print(f"Missing FTP link: {total - with_ftp}")

# Run analysis
analyze_summary('summary_output.tsv')


Total entries: 42086
Entries with FTP link: 12790
Missing FTP link: 29296


In [7]:
def count_fna_headers(fna_file):
    count = 0
    with open(fna_file, 'r') as file:
        for line in file:
            if line.startswith('>'):
                count += 1
    return count

def count_metadata_entries(tsv_file):
    with open(tsv_file, 'r', encoding='utf-8') as file:
        count = sum(1 for line in file) - 1  # Subtract header
    return count

def count_assembly_entries(assembly_file):
    count = 0
    with open(assembly_file, 'r') as file:
        for line in file:
            if not line.startswith('#'):
                count += 1
    return count

# File paths
fna_file = 'virushostdb.genomic.fna'
tsv_file = 'virushostdb.tsv'
assembly_file = 'assembly_summary_viral.txt'

# Results
fna_headers = count_fna_headers(fna_file)
tsv_entries = count_metadata_entries(tsv_file)
assembly_entries = count_assembly_entries(assembly_file)

print(f"Headers in .fna file: {fna_headers}")
print(f"Entries in virushostdb.tsv: {tsv_entries}")
print(f"Entries in assembly_summary_viral.txt: {assembly_entries}")


Headers in .fna file: 42086
Entries in virushostdb.tsv: 40098
Entries in assembly_summary_viral.txt: 14997


Why the differences?
1. Not all .fna sequence IDs are in virushostdb.tsv

You have 42,086 sequence IDs.
But only 32,951 were matched in the TSV file → meaning:
9,135 sequence IDs from .fna were not found in virushostdb.tsv.
This is common when .fna includes sequences not listed or outdated in the metadata.
2. Not all virus tax ids from TSV exist in assembly_summary_viral.txt

Of the 32,951 sequences matched from .fna → .tsv, only 12,790 had a matching tax ID in the assembly summary file → i.e., FTP path exists.
The rest (20,161) had no match in the assembly summary file → maybe:
Their virus tax ID is outdated.
They are not in the RefSeq/GenBank database (or their assemblies aren’t hosted).
Or the entries exist but lack FTP path.

In [8]:
def filter_summary_with_ftp(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w', newline='') as outfile:
        header = infile.readline()
        outfile.write(header)  # Copy header

        for line in infile:
            parts = line.strip().split('\t')
            if len(parts) >= 4 and parts[3] != 'N/A':
                outfile.write(line)

    print(f"Filtered summary written to: {output_file}")

# File paths
original_summary = 'summary_output.tsv'
filtered_summary = 'summary_with_ftp.tsv'

# Run filter
filter_summary_with_ftp(original_summary, filtered_summary)


Filtered summary written to: summary_with_ftp.tsv


In [9]:
import csv
import os

# Input/output file paths
summary_file = "summary_with_ftp.tsv"
output_file = "summary_with_filenames.tsv"

# Read the filtered summary and append the constructed filename
with open(summary_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', newline='') as outfile:
    reader = csv.reader(infile, delimiter='\t')
    writer = csv.writer(outfile, delimiter='\t')

    header = next(reader)
    writer.writerow(header + ['RefSeq Filename'])  # add new column

    for row in reader:
        ftp_path = row[3].strip()
        if ftp_path != 'N/A' and ftp_path != '':
            basename = os.path.basename(ftp_path)
            refseq_filename = f"{basename}_genomic.fna"
        else:
            refseq_filename = 'N/A'
        
        writer.writerow(row + [refseq_filename])

print(f"Generated: {output_file}")


Generated: summary_with_filenames.tsv


In [ ]:
GCF_000885895.1_ViralProj40309_genomic.fna
GCF_000885895.1_ViralProj40309_genomic.fna